
# Part 2 — DataFrames, Preprocessing, and Fast HDF5 Loader (Combined)

This notebook merges your **working HDF5 workflow** (OpenCV `imdecode` from per-image bytes in HDF5) with the new **cleaning, leak-safe split, transforms, and loaders** we built.


In [1]:

# --- Imports & config ---
from pathlib import Path
import numpy as np
import pandas as pd
import h5py, cv2, time, torch

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from tqdm.auto import tqdm

# Paths (adjust if needed)
DATA_DIR = Path("data")
TRAIN_META_CSV = DATA_DIR / "train-metadata.csv"
TEST_META_CSV  = DATA_DIR / "test-metadata.csv"
TRAIN_H5_PATH  = DATA_DIR / "train-image.hdf5"
TEST_H5_PATH   = DATA_DIR / "test-image.hdf5"

# Columns
ID_COL      = "isic_id"
TARGET_COL  = "target"
GROUP_COL   = "patient_id"

RANDOM_STATE = 42
N_SPLITS = 5

print(TRAIN_META_CSV.resolve())
print(TRAIN_H5_PATH.resolve())


C:\Users\alexs\Desktop\MASTER\DL\data\train-metadata.csv
C:\Users\alexs\Desktop\MASTER\DL\data\train-image.hdf5


c:\Users\alexs\miniconda3\envs\isic2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Inspect HDF5 structure (per-image bytes expected)

In [2]:

def inspect_h5_keys(path, limit=5):
    with h5py.File(path, "r") as f:
        keys = list(f.keys())
        print(f"{path.name}: {len(keys)} keys")
        for k in keys[:limit]:
            ds = f[k]
            shape = getattr(ds, "shape", None)
            dtype = getattr(ds, "dtype", None)
            print("  key:", k, "| shape:", shape, "| dtype:", dtype)
        if keys:
            sample = keys[0]
            arr = f[sample][()]  # raw bytes/uint8 array
            print("Sample entry type:", type(arr), "len:", len(arr) if hasattr(arr, "__len__") else None)

inspect_h5_keys(TRAIN_H5_PATH)


train-image.hdf5: 400959 keys
  key: ISIC_0015670 | shape: (5883,) | dtype: uint8
  key: ISIC_0015845 | shape: (5204,) | dtype: uint8
  key: ISIC_0015864 | shape: (6257,) | dtype: uint8
  key: ISIC_0015902 | shape: (3303,) | dtype: uint8
  key: ISIC_0024200 | shape: (5195,) | dtype: uint8
Sample entry type: <class 'numpy.ndarray'> len: 5883


## Load & normalize metadata

In [3]:

def _read_meta(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    # Harmonize id/target column names if needed
    if ID_COL not in df.columns:
        for alt in ["image_id", "img_id", "image_name", "id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: ID_COL})
                print(f"[INFO] Renamed '{alt}' -> '{ID_COL}'")
                break
    if TARGET_COL not in df.columns:
        for alt in ["label", "malignant", "cancer", "diagnosis", "target_binary"]:
            if alt in df.columns:
                df = df.rename(columns={alt: TARGET_COL})
                print(f"[INFO] Renamed '{alt}' -> '{TARGET_COL}'")
                break
    return df

meta_df = _read_meta(TRAIN_META_CSV)
test_meta_df = _read_meta(TEST_META_CSV) if TEST_META_CSV.exists() else None

print("Train meta cols:", meta_df.columns.tolist()[:12])
print(meta_df.head(3))


Train meta cols: ['isic_id', 'target', 'patient_id', 'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm', 'image_type', 'tbp_tile_type', 'tbp_lv_a', 'tbp_lv_aext', 'tbp_lv_b']
        isic_id  target  patient_id  age_approx   sex anatom_site_general  \
0  ISIC_0015670       0  IP_1235828        60.0  male     lower extremity   
1  ISIC_0015845       0  IP_8170065        60.0  male           head/neck   
2  ISIC_0015864       0  IP_6724798        60.0  male     posterior torso   

   clin_size_long_diam_mm          image_type tbp_tile_type   tbp_lv_a  ...  \
0                    3.04  TBP tile: close-up     3D: white  20.244422  ...   
1                    1.10  TBP tile: close-up     3D: white  31.712570  ...   
2                    3.40  TBP tile: close-up        3D: XP  22.575830  ...   

    lesion_id  iddx_full  iddx_1  iddx_2  iddx_3  iddx_4  iddx_5  \
0         NaN     Benign  Benign     NaN     NaN     NaN     NaN   
1  IL_6727506     Benign  Benign     NaN     

C:\Users\alexs\AppData\Local\Temp\ipykernel_22388\1781761395.py:2: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


## Basic cleaning

In [4]:

df = meta_df.copy()

# Drop dupe IDs
df = df.drop_duplicates(subset=[ID_COL])

# Coerce types & normalize categories
for col in ["age_approx", "age"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").clip(lower=0, upper=100)
        df.rename(columns={col: "age_approx"}, inplace=True)
        break

if "sex" in df.columns:
    df["sex"] = (df["sex"].astype(str).str.lower().str.strip()
                 .replace({"male":"male","m":"male","female":"female","f":"female","nan":np.nan})
                )

for site_col in ["anatom_site_general", "anatom_site_general_challenge", "anatom_site"]:
    if site_col in df.columns:
        df[site_col] = df[site_col].astype(str).str.lower().str.strip()
        if site_col != "anatom_site_general":
            df.rename(columns={site_col: "anatom_site_general"}, inplace=True)
        break

# Handle missings
if "sex" in df.columns:
    df["sex"] = df["sex"].fillna("unknown")
if "anatom_site_general" in df.columns:
    df["anatom_site_general"] = df["anatom_site_general"].fillna("unknown")
if "age_approx" in df.columns:
    df["age_approx"] = df.groupby("sex")["age_approx"].transform(lambda s: s.fillna(s.median()))
    df["age_approx"] = df["age_approx"].fillna(df["age_approx"].median())

# Ensure target exists
if TARGET_COL in df.columns:
    df = df[~df[TARGET_COL].isna()].copy()
    df[TARGET_COL] = df[TARGET_COL].astype(int)
else:
    print(f"[WARN] '{TARGET_COL}' not found — continuing test-only.")

print(df.isna().sum().sort_values(ascending=False).head(10))
df.head(3)

iddx_5               400958
mel_mitotic_index    400916
mel_thick_mm         400908
iddx_4               400440
iddx_3               399944
iddx_2               399941
lesion_id            378954
image_type                0
tbp_tile_type             0
tbp_lv_a                  0
dtype: int64


,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_a,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,0,IP_6724798,60.0,male,posterior torso,3.40,TBP tile: close-up,3D: XP,22.575830,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040


## Quick EDA

In [5]:

def pct(x): 
    return 100.0 * x / x.sum() if x.sum() else np.nan

if TARGET_COL in df.columns:
    vc = df[TARGET_COL].value_counts().sort_index()
    print("Class counts:\n", vc)
    print("Class %:\n", pct(vc).round(2))

for col in ["sex","anatom_site_general"]:
    if col in df.columns:
        print(f"\n{col} distribution:")
        print(df[col].value_counts().head(10))

if "age_approx" in df.columns:
    print("\nAge summary:")
    print(df["age_approx"].describe(percentiles=[.1,.25,.5,.75,.9]).round(2))

Class counts:
 target
0    400616
1       343
Name: count, dtype: int64
Class %:
 target
0    99.91
1     0.09
Name: count, dtype: float64

sex distribution:
sex
male       265484
female     123962
unknown     11513
Name: count, dtype: int64

anatom_site_general distribution:
anatom_site_general
posterior torso    121868
lower extremity    103008
anterior torso      87749
upper extremity     70541
head/neck           12037
nan                  5756
Name: count, dtype: int64

Age summary:
count    400959.00
mean         57.99
std          13.55
min           5.00
10%          40.00
25%          50.00
50%          60.00
75%          70.00
90%          75.00
max          85.00
Name: age_approx, dtype: float64


## Leak-safe train/val split

In [6]:

if TARGET_COL not in df.columns:
    print("[INFO] No labels — skipping split.")
    train_df, val_df = df.copy(), pd.DataFrame(columns=df.columns)
else:
    use_groups = GROUP_COL in df.columns and df[GROUP_COL].notna().any()
    X = df[ID_COL]; y = df[TARGET_COL]
    if use_groups:
        groups = df[GROUP_COL].astype(str)
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        tr_idx, va_idx = list(sgkf.split(X, y, groups))[0]
        print("[INFO] Using StratifiedGroupKFold (no patient leakage)")
    else:
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        tr_idx, va_idx = list(skf.split(X, y))[0]
        print("[WARN] No groups — using StratifiedKFold")
    train_df = df.iloc[tr_idx].reset_index(drop=True)
    val_df   = df.iloc[va_idx].reset_index(drop=True)
    print(f"train: {len(train_df):,}   val: {len(val_df):,}")
    print("train %:\n", pct(train_df[TARGET_COL].value_counts()).round(2))
    print("val %:\n",   pct(val_df[TARGET_COL].value_counts()).round(2))

[INFO] Using StratifiedGroupKFold (no patient leakage)
train: 323,700   val: 77,259
train %:
 target
0    99.92
1     0.08
Name: count, dtype: float64
val %:
 target
0    99.91
1     0.09
Name: count, dtype: float64


## Torchvision transforms

In [7]:

IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.1),
    T.RandomRotation(degrees=10),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transforms = val_transforms

## Fast HDF5 Dataset (per-image bytes + OpenCV decode)

In [8]:

class ISIC_HDF5_Dataset(Dataset):
    """Loads JPEG/PNG bytes from HDF5 (keys = isic_id), decodes with OpenCV, applies transforms."""
    def __init__(self, df: pd.DataFrame, hdf5_path: str, transform=None, is_labelled: bool = True):
        self.df = df.reset_index(drop=True)
        self.hdf5_path = hdf5_path
        self.transform = transform
        self.is_labelled = is_labelled

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        isic_id = row[ID_COL]
        image = self._load_image_from_hdf5(isic_id)  # np.ndarray RGB HxWx3 (uint8)
        # PIL-free pipeline: convert to torch and normalize via transforms pipeline
        # Our transforms expect PIL.Image, so convert to PIL only here
        from PIL import Image
        pil = Image.fromarray(image)  # RGB
        if self.transform is not None:
            img_t = self.transform(pil)
        else:
            import torchvision.transforms.functional as F
            img_t = F.to_tensor(pil)

        if self.is_labelled and (TARGET_COL in self.df.columns):
            y = int(row[TARGET_COL])
            return img_t, torch.tensor(y).float(), isic_id
        else:
            return img_t, isic_id

    def _load_image_from_hdf5(self, isic_id: str):
        """Read raw image bytes by key and decode with OpenCV."""
        # Open the file per call: in practice this is often faster/safer with HDF5 + DataLoader workers
        # and matches your previous fast workflow.
        with h5py.File(self.hdf5_path, "r") as hf:
            data = hf[isic_id][()]  # bytes as np array (uint8) or Python bytes
        if isinstance(data, bytes):
            encoded = np.frombuffer(data, dtype=np.uint8)
        else:
            encoded = np.asarray(data, dtype=np.uint8).reshape(-1)
        img_bgr = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise ValueError(f"Failed to decode image for id={isic_id}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return img_rgb


## DataLoaders

In [9]:

BATCH_SIZE = 64

train_ds = ISIC_HDF5_Dataset(train_df, str(TRAIN_H5_PATH), transform=train_transforms, is_labelled=True)
val_ds   = ISIC_HDF5_Dataset(val_df,   str(TRAIN_H5_PATH), transform=val_transforms,   is_labelled=True)
test_ds  = ISIC_HDF5_Dataset(test_meta_df if test_meta_df is not None else df.head(10),
                             str(TEST_H5_PATH), transform=test_transforms, is_labelled=False)

# IMPORTANT: many HDF5 setups prefer num_workers=0 to avoid contention
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Created train/val/test loaders.")


Created train/val/test loaders.


# Backbone

In [10]:
# === ResNet50 model + training/eval utilities ===
import math, time, torch, torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torchvision.models import resnet50
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----- Model -----
model = resnet50(weights="IMAGENET1K_V2")  # torchvision>=0.13
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 1)       # binary head
model = model.to(device)

# ----- Loss (handles class imbalance via pos_weight) -----
# Compute pos_weight = (N_neg / N_pos) from the current training split
if TARGET_COL in train_df.columns:
    n_pos = (train_df[TARGET_COL] == 1).sum()
    n_neg = (train_df[TARGET_COL] == 0).sum()
    if n_pos > 0:
        pos_weight_val = max(1.0, n_neg / max(1, n_pos))
    else:
        pos_weight_val = 1.0
else:
    pos_weight_val = 1.0

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], device=device))

# ----- Optimizer & Scheduler -----
lr = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
# Cosine decay over num_epochs; will set T_max later after you choose epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1, eta_min=1e-6)

scaler = GradScaler(enabled=torch.cuda.is_available())

def train_one_epoch(model, loader, optimizer, scaler, epoch, log_every=50):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for i, batch in enumerate(loader):
        if isinstance(batch, (list, tuple)) and len(batch) == 3:
            xb, yb, _ids = batch
        else:
            # if your dataset returns (x, id) for some reason
            xb, yb = batch[0], None

        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True).view(-1, 1)  # (B,1) for BCE

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        running_loss += loss.item() * bs
        n += bs

        if (i + 1) % log_every == 0:
            print(f"Epoch {epoch} | Step {i+1}/{len(loader)} | "
                  f"Loss {running_loss/n:.4f} | LR {optimizer.param_groups[0]['lr']:.2e}")

    dt = time.time() - t0
    return running_loss / max(1, n), dt

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    n = 0
    all_labels = []
    all_probs  = []
    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 3:
            xb, yb, _ids = batch
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True).view(-1, 1)
        else:
            # evaluation requires labels; if absent, just return loss=None/AUC=None
            return None, None

        logits = model(xb)
        loss = criterion(logits, yb)

        bs = xb.size(0)
        running_loss += loss.item() * bs
        n += bs

        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        labels = yb.detach().cpu().numpy().ravel()
        all_probs.append(probs)
        all_labels.append(labels)

    import numpy as np
    all_probs  = np.concatenate(all_probs) if all_probs else np.array([])
    all_labels = np.concatenate(all_labels) if all_labels else np.array([])

    val_loss = running_loss / max(1, n)
    try:
        val_auc = roc_auc_score(all_labels, all_probs) if all_labels.size and np.unique(all_labels).size > 1 else None
    except Exception:
        val_auc = None
    return val_loss, val_auc


Device: cuda


C:\Users\alexs\AppData\Local\Temp\ipykernel_22388\3266217833.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())


## Sanity check: one batch

In [ ]:

batch = next(iter(train_loader))
if isinstance(batch, (list, tuple)) and len(batch) == 3:
    xb, yb, ids = batch
    print("Batch images:", xb.shape, "Batch targets:", yb.shape)
    print("First 5 ids:", list(ids[:5]))
else:
    xb, ids = batch
    print("Batch images:", xb.shape, "First 5 ids:", list(ids[:5]))


# Train

In [ ]:
# === Install & setup MLflow ===
!pip install -q mlflow

import mlflow
import mlflow.pytorch

mlflow.set_tracking_uri("file:./mlruns")  # local logging folder
mlflow.set_experiment("isic_resnet50")    # creates/uses experiment


In [24]:
# === Train! ===
num_epochs = 5  # start small for a smoke test; then try 10–20
# update scheduler T_max to match planned epochs
for _ in range(scheduler.last_epoch + 1):  # reset if needed
    pass
scheduler.T_max = num_epochs

best_auc = -math.inf
best_state = None

for epoch in range(1, num_epochs + 1):
    train_loss, train_dt = train_one_epoch(model, train_loader, optimizer, scaler, epoch, log_every=50)
    val_loss, val_auc = evaluate(model, val_loader)

    scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"\nEpoch {epoch}/{num_epochs} "
          f"| train_loss {train_loss:.4f} ({train_dt:.1f}s) "
          f"| val_loss {val_loss if val_loss is not None else 'NA'} "
          f"| val_auc {f'{val_auc:.4f}' if val_auc is not None else 'NA'} "
          f"| lr {lr_now:.2e}\n")

    # Track best by AUC (fallback to lowest val_loss if AUC unavailable)
    score = (val_auc if val_auc is not None else -val_loss)
    if score is not None and score > best_auc:
        best_auc = score
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Save best
if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), "resnet50_isic_best.pth")
print("Saved best weights -> resnet50_isic_best.pth")


C:\Users\alexs\AppData\Local\Temp\ipykernel_10816\3266217833.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 1 | Step 50/5058 | Loss 14.6660 | LR 3.00e-04
Epoch 1 | Step 100/5058 | Loss 14.4440 | LR 3.00e-04
Epoch 1 | Step 150/5058 | Loss 11.4175 | LR 3.00e-04
Epoch 1 | Step 200/5058 | Loss 9.0665 | LR 3.00e-04
Epoch 1 | Step 250/5058 | Loss 7.4299 | LR 3.00e-04
Epoch 1 | Step 300/5058 | Loss 6.4197 | LR 3.00e-04
Epoch 1 | Step 350/5058 | Loss 5.7546 | LR 3.00e-04
Epoch 1 | Step 400/5058 | Loss 5.1495 | LR 3.00e-04
Epoch 1 | Step 450/5058 | Loss 4.7181 | LR 3.00e-04
Epoch 1 | Step 500/5058 | Loss 4.7595 | LR 3.00e-04
Epoch 1 | Step 550/5058 | Loss 4.4630 | LR 3.00e-04
Epoch 1 | Step 600/5058 | Loss 4.3318 | LR 3.00e-04
Epoch 1 | Step 650/5058 | Loss 4.1141 | LR 3.00e-04
Epoch 1 | Step 700/5058 | Loss 3.8725 | LR 3.00e-04
Epoch 1 | Step 750/5058 | Loss 3.8367 | LR 3.00e-04
Epoch 1 | Step 800/5058 | Loss 3.6982 | LR 3.00e-04
Epoch 1 | Step 850/5058 | Loss 3.5429 | LR 3.00e-04
Epoch 1 | Step 900/5058 | Loss 3.5202 | LR 3.00e-04
Epoch 1 | Step 950/5058 | Loss 3.3921 | LR 3.00e-04
Epoch 1 | 

C:\Users\alexs\AppData\Local\Temp\ipykernel_10816\3266217833.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 2 | Step 50/5058 | Loss 1.3135 | LR 2.71e-04
Epoch 2 | Step 100/5058 | Loss 1.1838 | LR 2.71e-04
Epoch 2 | Step 150/5058 | Loss 1.2141 | LR 2.71e-04
Epoch 2 | Step 200/5058 | Loss 1.3404 | LR 2.71e-04
Epoch 2 | Step 250/5058 | Loss 1.3449 | LR 2.71e-04
Epoch 2 | Step 300/5058 | Loss 1.2611 | LR 2.71e-04
Epoch 2 | Step 350/5058 | Loss 1.4674 | LR 2.71e-04
Epoch 2 | Step 400/5058 | Loss 1.4547 | LR 2.71e-04
Epoch 2 | Step 450/5058 | Loss 1.4622 | LR 2.71e-04
Epoch 2 | Step 500/5058 | Loss 1.4445 | LR 2.71e-04
Epoch 2 | Step 550/5058 | Loss 1.4145 | LR 2.71e-04
Epoch 2 | Step 600/5058 | Loss 1.4544 | LR 2.71e-04
Epoch 2 | Step 650/5058 | Loss 1.4309 | LR 2.71e-04
Epoch 2 | Step 700/5058 | Loss 1.4093 | LR 2.71e-04
Epoch 2 | Step 750/5058 | Loss 1.3919 | LR 2.71e-04
Epoch 2 | Step 800/5058 | Loss 1.4309 | LR 2.71e-04
Epoch 2 | Step 850/5058 | Loss 1.3853 | LR 2.71e-04
Epoch 2 | Step 900/5058 | Loss 1.4024 | LR 2.71e-04
Epoch 2 | Step 950/5058 | Loss 1.4160 | LR 2.71e-04
Epoch 2 | Ste

C:\Users\alexs\AppData\Local\Temp\ipykernel_10816\3266217833.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 3 | Step 50/5058 | Loss 2.1294 | LR 1.97e-04
Epoch 3 | Step 100/5058 | Loss 1.8765 | LR 1.97e-04
Epoch 3 | Step 150/5058 | Loss 1.6145 | LR 1.97e-04
Epoch 3 | Step 200/5058 | Loss 1.5897 | LR 1.97e-04
Epoch 3 | Step 250/5058 | Loss 1.4988 | LR 1.97e-04
Epoch 3 | Step 300/5058 | Loss 1.3899 | LR 1.97e-04
Epoch 3 | Step 350/5058 | Loss 1.3303 | LR 1.97e-04
Epoch 3 | Step 400/5058 | Loss 1.3673 | LR 1.97e-04
Epoch 3 | Step 450/5058 | Loss 1.3093 | LR 1.97e-04
Epoch 3 | Step 500/5058 | Loss 1.3076 | LR 1.97e-04
Epoch 3 | Step 550/5058 | Loss 1.3249 | LR 1.97e-04
Epoch 3 | Step 600/5058 | Loss 1.4092 | LR 1.97e-04
Epoch 3 | Step 650/5058 | Loss 1.4084 | LR 1.97e-04
Epoch 3 | Step 700/5058 | Loss 1.3712 | LR 1.97e-04
Epoch 3 | Step 750/5058 | Loss 1.4102 | LR 1.97e-04
Epoch 3 | Step 800/5058 | Loss 1.4690 | LR 1.97e-04
Epoch 3 | Step 850/5058 | Loss 1.4533 | LR 1.97e-04
Epoch 3 | Step 900/5058 | Loss 1.4854 | LR 1.97e-04
Epoch 3 | Step 950/5058 | Loss 1.4919 | LR 1.97e-04
Epoch 3 | Ste

C:\Users\alexs\AppData\Local\Temp\ipykernel_10816\3266217833.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 4 | Step 50/5058 | Loss 0.5997 | LR 1.04e-04
Epoch 4 | Step 100/5058 | Loss 0.9566 | LR 1.04e-04
Epoch 4 | Step 150/5058 | Loss 1.3698 | LR 1.04e-04
Epoch 4 | Step 200/5058 | Loss 1.5522 | LR 1.04e-04
Epoch 4 | Step 250/5058 | Loss 1.6149 | LR 1.04e-04
Epoch 4 | Step 300/5058 | Loss 1.6172 | LR 1.04e-04
Epoch 4 | Step 350/5058 | Loss 1.5443 | LR 1.04e-04
Epoch 4 | Step 400/5058 | Loss 1.7062 | LR 1.04e-04
Epoch 4 | Step 450/5058 | Loss 1.6191 | LR 1.04e-04
Epoch 4 | Step 500/5058 | Loss 1.5639 | LR 1.04e-04
Epoch 4 | Step 550/5058 | Loss 1.5230 | LR 1.04e-04
Epoch 4 | Step 600/5058 | Loss 1.4971 | LR 1.04e-04
Epoch 4 | Step 650/5058 | Loss 1.5825 | LR 1.04e-04
Epoch 4 | Step 700/5058 | Loss 1.6763 | LR 1.04e-04
Epoch 4 | Step 750/5058 | Loss 1.6280 | LR 1.04e-04
Epoch 4 | Step 800/5058 | Loss 1.6208 | LR 1.04e-04
Epoch 4 | Step 850/5058 | Loss 1.6142 | LR 1.04e-04
Epoch 4 | Step 900/5058 | Loss 1.6052 | LR 1.04e-04
Epoch 4 | Step 950/5058 | Loss 1.6085 | LR 1.04e-04
Epoch 4 | Ste

C:\Users\alexs\AppData\Local\Temp\ipykernel_10816\3266217833.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 5 | Step 50/5058 | Loss 1.3830 | LR 2.96e-05
Epoch 5 | Step 100/5058 | Loss 1.3875 | LR 2.96e-05
Epoch 5 | Step 150/5058 | Loss 1.3090 | LR 2.96e-05
Epoch 5 | Step 200/5058 | Loss 1.3875 | LR 2.96e-05
Epoch 5 | Step 250/5058 | Loss 1.4336 | LR 2.96e-05
Epoch 5 | Step 300/5058 | Loss 1.5103 | LR 2.96e-05
Epoch 5 | Step 350/5058 | Loss 1.6127 | LR 2.96e-05
Epoch 5 | Step 400/5058 | Loss 2.4837 | LR 2.96e-05
Epoch 5 | Step 450/5058 | Loss 2.3915 | LR 2.96e-05
Epoch 5 | Step 500/5058 | Loss 2.2432 | LR 2.96e-05
Epoch 5 | Step 550/5058 | Loss 2.1898 | LR 2.96e-05
Epoch 5 | Step 600/5058 | Loss 2.1433 | LR 2.96e-05
Epoch 5 | Step 650/5058 | Loss 2.1420 | LR 2.96e-05
Epoch 5 | Step 700/5058 | Loss 2.0713 | LR 2.96e-05
Epoch 5 | Step 750/5058 | Loss 2.0123 | LR 2.96e-05
Epoch 5 | Step 800/5058 | Loss 1.9277 | LR 2.96e-05
Epoch 5 | Step 850/5058 | Loss 1.8954 | LR 2.96e-05
Epoch 5 | Step 900/5058 | Loss 1.8398 | LR 2.96e-05
Epoch 5 | Step 950/5058 | Loss 1.8028 | LR 2.96e-05
Epoch 5 | Ste

In [ ]:
# === Train with MLflow logging ===
num_epochs = 5  # or more
import mlflow

mlflow.set_tracking_uri(uri="http://<host>:<port>")
mlflow.start_run(run_name="resnet50-baseline")

# Log hyperparams once
mlflow.log_param("backbone", "resnet50")
mlflow.log_param("epochs", num_epochs)
mlflow.log_param("batch_size", BATCH_SIZE)
mlflow.log_param("lr", lr)
mlflow.log_param("optimizer", "AdamW")
mlflow.log_param("scheduler", "CosineAnnealingLR")

best_auc = -math.inf
best_state = None

for epoch in range(1, num_epochs + 1):
    train_loss, train_dt = train_one_epoch(model, train_loader, optimizer, scaler, epoch, log_every=50)
    val_loss, val_auc = evaluate(model, val_loader)

    scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch}/{num_epochs} "
          f"| train_loss {train_loss:.4f} "
          f"| val_loss {val_loss if val_loss is not None else 'NA'} "
          f"| val_auc {f'{val_auc:.4f}' if val_auc is not None else 'NA'} "
          f"| lr {lr_now:.2e}")

    # --- MLflow logging ---
    mlflow.log_metric("train_loss", train_loss, step=epoch)
    if val_loss is not None:
        mlflow.log_metric("val_loss", val_loss, step=epoch)
    if val_auc is not None:
        mlflow.log_metric("val_auc", val_auc, step=epoch)
    mlflow.log_metric("lr", lr_now, step=epoch)

    # Save best
    score = (val_auc if val_auc is not None else -val_loss)
    if score is not None and score > best_auc:
        best_auc = score
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Save model locally + to MLflow
if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), "resnet50_isic_best.pth")
print("Saved best weights -> resnet50_isic_best.pth")

mlflow.pytorch.log_model(model, artifact_path="model")
mlflow.end_run()


# Inference on Test Set & Submission

In [11]:
# -----------------------------
# 6. Load trained ResNet50 from models/resnet50_isic_best.pth
# -----------------------------
import os
import torch
import torch.nn as nn
from torchvision import models

ckpt_path = "models/resnet50_isic_best.pth"
assert os.path.exists(ckpt_path), f"Checkpoint not found at {ckpt_path}"

# 1) Build the SAME architecture you trained
#    (adjust num classes if needed; here it's binary with a single logit)
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

# 2) Load checkpoint robustly
ckpt = torch.load(ckpt_path, map_location=device)

# If the whole model was saved (rare), handle it:
if isinstance(ckpt, nn.Module):
    model = ckpt.to(device)
else:
    # Try to find the actual state_dict inside typical wrappers
    if isinstance(ckpt, dict):
        if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
            state_dict = ckpt["state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            state_dict = ckpt["model"]
        elif "model_state" in ckpt and isinstance(ckpt["model_state"], dict):
            state_dict = ckpt["model_state"]
        else:
            # maybe ckpt is already a state_dict
            state_dict = ckpt
    else:
        state_dict = ckpt

    # Strip common prefixes (e.g., 'module.' from DataParallel, or 'model.')
    clean_state_dict = {}
    for k, v in state_dict.items():
        new_k = k
        if new_k.startswith("module."):
            new_k = new_k[len("module."):]
        if new_k.startswith("model."):
            new_k = new_k[len("model."):]
        clean_state_dict[new_k] = v

    missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
    if missing:
        print("Missing keys:", missing)
    if unexpected:
        print("Unexpected keys:", unexpected)

model.eval()
print("Loaded weights from", ckpt_path)


C:\Users\alexs\AppData\Local\Temp\ipykernel_22388\2736987092.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


Loaded weights from models/resnet50_isic_best.pth


In [12]:
# -----------------------------
# 7. Inference on Test Set & Submission
# -----------------------------
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

model.eval()

all_rows = []  # list of dicts: {"isic_id": str, "target": float}

use_amp = torch.cuda.is_available()  # mixed precision if on GPU

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference on Test"):
        # Expecting test_loader to yield (images, isic_ids)
        images, isic_ids = batch
        images = images.to(device, non_blocking=True)

        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)  # shape: [B] or [B, 1]
        else:
            logits = model(images)

        # normalize shape to [B]
        if logits.ndim > 1:
            logits = logits.squeeze(-1)

        # convert logits -> probabilities if needed
        probs = torch.sigmoid(logits)

        # to cpu numpy
        probs = probs.float().detach().cpu().numpy()
        # clip to [0, 1] (safety)
        probs = np.clip(probs, 0.0, 1.0)

        for isic_id, p in zip(isic_ids, probs):
            # ensure plain python types
            all_rows.append({"isic_id": str(isic_id), "target": float(p)})

# Build dataframe from collected predictions
pred_df = pd.DataFrame(all_rows)

# If you have a sample_submission.csv, align to its order (recommended for Kaggle)
sample_sub_path = "sample_submission.csv"
if os.path.exists(sample_sub_path):
    sample_df = pd.read_csv(sample_sub_path)
    # map predictions by id
    pred_map = {row["isic_id"]: row["target"] for _, row in pred_df.iterrows()}
    # fill in any missing ids with 0.0 (or another fallback)
    sample_df["target"] = sample_df["isic_id"].map(pred_map).fillna(0.0).astype(float)
    submission_df = sample_df
else:
    # otherwise just sort by isic_id for determinism
    submission_df = pred_df.sort_values("isic_id").reset_index(drop=True)

# Final safety: ensure valid column order/types
submission_df = submission_df[["isic_id", "target"]]
submission_df["isic_id"] = submission_df["isic_id"].astype(str)
submission_df["target"] = submission_df["target"].astype(float)

submission_file = "submission.csv"
submission_df.to_csv(submission_file, index=False)

print(f"Saved submission with {len(submission_df)} rows to {submission_file}")
display(submission_df.head(10))


Inference on Test:   0%|          | 0/2 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 6572, 2168) exited unexpectedly


### Notes
- This **HDF5-bytes + OpenCV** path mirrors your smooth workflow and avoids the per-index random-slice overhead.
- We use `num_workers=0` to prevent HDF5 contention. If you see headroom, try `num_workers=2` and compare throughput.
- If the first batch is slow but subsequent batches are fast, that's expected due to OS caching.
